# Compare metadata models: deduplicated Subset data

We compare five ways to predict malignancy from age, sex and anatomical site, the metadata recorded alongside each image. The questions are which model ranks malignant images best and which fields add useful information.

[03_patient_fold_verification.ipynb](03_patient_fold_verification.ipynb) supplies shared patient groups, or folds. [04_metadata_baselines.ipynb](04_metadata_baselines.ipynb) introduces the two simplest references. Every method here is assessed on patients excluded from its own training.

The saved run favours spline logistic regression, which learns a smooth age relationship: average precision (AP) 0.0492 pooled (0.0648 mean-fold) versus 0.0184 pooled (0.0189 mean-fold) for the dummy. These are development comparisons, before adding images; the reserved test is not used.

For metric definitions, see the [short concept guide](../concepts.md).


**Deduplicated workflow adaptation.** This notebook uses the frozen `dedup_v1` development manifest and original patient assignments. It reads saved clean artifacts; executing it starts no training and does not load the reserved test. The original notebook is preserved as the presentation template: [05_metadata_models.ipynb](../Subset/05_metadata_models.ipynb).

The clean workflow's primary ranking measure is **unweighted mean outer-fold AP**. The original pooled tables and figures are retained for presentation consistency and are explicitly secondary; their values are not interchangeable with mean-fold scores.

**Current reporting priority:** primary unweighted mean outer-fold ROC-AUC, followed by secondary AP, log loss and Brier. Pooled curves support the comparison. Verified versioned baseline diagnostics supply the dummy/logistic predictions; stage 05 uses matching fixed-family version-2 predictions. Original AP-era numerical examples, ablations, age illustrations and exploratory cutoffs below remain clearly historical supporting evidence. No all-development fit is recreated.

## Context and methods

We predict the outcome of each image, while keeping related images from one patient together. For each fold, train on the other four and predict the held-out group. Combining the five groups gives out-of-fold (OOF) predictions: each scored patient was excluded from their model's training.

Preprocessing, converting raw fields into model inputs, is fitted inside training. Validation patients cannot change categories, missing-age replacements or the learned age curve. Exclude diagnosis and benign/malignant descriptions because they reveal the answer, plus identifiers and fold numbers.

Only 1.89% of images are malignant. Average precision (AP), the historical selection measure, rewards placing these rare cases near the top of a score ranking. The results also explain ROC-AUC and probability-error measures. An output in [0,1] is a probability estimate, not proof that its numerical risk is accurate.

Confusion-matrix cutoffs are chosen and assessed from the same OOF labels, so they illustrate exploratory trade-offs. [08_metadata_robustness.ipynb](08_metadata_robustness.ipynb) separates cutoff selection from assessment.


### What each model contributes

| Model | Why include it? | What it learns |
|---|---|---|
| Prevalence dummy | Set a reference for a model with no useful ranking | The malignant-image rate in its training data |
| Standard logistic regression | Start with a simple, readable model | Separate effects for each age category, sex and site |
| Spline logistic regression | Let nearby ages share a smooth pattern | A smooth age curve, plus sex and site effects |
| Random forest | Check whether combinations of fields help | Many decision trees whose predictions are averaged |
| XGBoost | Check a second way to combine decision trees | Trees fitted in sequence to improve earlier predictions |

The spline model uses four knots, which are points where its age curve can bend. This gives age some flexibility without fitting a separate effect to every age category. These settings were chosen as a starting specification; this notebook does not search for the best settings.


## Data

### Set paths and define the modelling tools

The following cells define the models and how we check them. All five methods read the same development data and use the same patient folds, so differences in results reflect the model choices.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src/melanoma_detection").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from melanoma_detection.metadata_tuning_v2_reporting import (
    load_tuning, load_baseline_report, render_section, render_baseline_section,
    display_historical_cell, display_historical_source, original_snapshot,
)
plt.rcdefaults()
plt.rcParams['figure.dpi'] = 120
ORIGINAL_EXPORT_DPI = 320
NN_COMPARISON_COLORS = ['#2563EB', '#D97706', '#D97706', '#D97706', '#303030']
report = load_baseline_report(PROJECT_ROOT, 'dedup_v1', '05', run_id='roc_auc_grid36_v1')
report["export_dpi"] = ORIGINAL_EXPORT_DPI
display(Markdown("**Version-2 saved-results report:** " + report["status"] +
    ". ROC-AUC is primary; AP and probability errors are secondary. No training, final fitting or reserved-test loading starts here."))


### Define the shared preprocessing

The standard models turn each category into a yes/no column, a process called one-hot encoding. For example, a site column records whether an image comes from the torso. Age is also treated as a category for these comparisons.

The spline model treats age as a number and learns a smooth curve. It fills a missing age with the training median and scales the curve's numerical features. Each training fold learns its own transformation, using only its training patients.


In [ ]:
display_historical_cell(PROJECT_ROOT, 'dedup_v1', '05', 7)

### Define how we generate held-out predictions

The separate clean runner created a fresh copy of each model for every fold, fit it on four folds and predict the fifth. This report reads verified saved predictions and aligns them with retained image identities; it does not refit models. The next cell checks that no patient appears in more than one fold.


In [ ]:
display_historical_cell(PROJECT_ROOT, 'dedup_v1', '05', 9)

### Load and check the development data

Before reading the saved fits, check that the required fields are present, the target contains both classes and every patient stays in one fold. The counts below describe the data used for every comparison.


In [ ]:
render_baseline_section(report, 'inputs')

## Results

### 1. Load patient-safe out-of-fold predictions

In the completed clean run, each model trained on the other four folds and predicted the held-out fold. Repeating this process gives one validation probability for every image while keeping patients separated.


In [ ]:
render_baseline_section(report, 'inputs')

### 2. Compare probability performance

The table compares all five models on their combined held-out predictions. ROC-AUC and AP are better when larger; log loss and Brier score are better when smaller. Use the ROC-AUC-first mean-fold table first; pooled metrics retain the original descriptive view. Use primary mean-fold ROC-AUC first, then secondary AP, then check whether the other measures tell a consistent story.


In [ ]:
render_baseline_section(report, 'metrics')

### How to read the scores

- Average precision (AP) rewards placing malignant images near the top of the risk ranking. It is not the percentage of images classified correctly. The dummy's 0.0184 provides a reference close to the malignant-image rate.
- ROC-AUC describes how often a malignant image gets a higher score than a benign image. A value of 0.5 is chance-level ranking; 1 is perfect ranking.
- Log loss penalises incorrect probability estimates, especially confident mistakes. Smaller is better.
- Brier score is the average squared difference between the probability and the actual 0/1 label. Smaller is better. With many benign images, a small Brier score alone can hide poor detection of malignant cases.

The spline leads on pooled and mean-fold AP and on pooled ROC-AUC among these fixed models. Its primary mean-fold ROC-AUC is 0.0648. Its AP of 0.0492 is still low, so its higher score does not make metadata a reliable standalone diagnostic method.


### 3. Check how results vary across patient folds

Each row below is a different held-out patient group. Compare the same model across rows before judging its pooled score. Large changes suggest that the result depends on which patients happen to be held out.


In [ ]:
render_baseline_section(report, 'fold_metrics')

The spline's AP ranges from 0.0373 to 0.1122 across folds. Different patient groups produce different results, and the spline does not lead every fold. This variation is a reason to check uncertainty in later model comparisons.


### 4. Read the ROC and precision-recall curves

Each row is one model: blue logistic regression, gold spline, olive random forest and orange XGBoost. Curves trace changing cutoffs; axis fractions such as 0.2 mean 20%.

- **Left: ROC.** The horizontal axis is benign images incorrectly flagged (false-positive rate); the vertical axis is malignant images found (recall). Closer to the upper-left is better. The grey diagonal is chance ranking.
- **Right: precision-recall.** Recall is horizontal; precision, the malignant share of flagged images, is vertical. Higher precision at the same recall means fewer false alarms. The grey line marks the 1.89% malignant-image share. The vertical view stops at 30%, excluding the endpoint at recall zero and precision one.

Read both panels: above-chance ranking can coexist with low precision. These curves compare cutoffs without endorsing one.


**Historical supporting section:** its saved pre-migration diagnostic is retained below; it does not select the current final procedure.

In [ ]:
render_baseline_section(report, 'curves')

### 5. Remove fields to see where the signal comes from

The separate clean runner refitted the same spline-logistic model using different combinations of age, sex and site. This is called feature ablation. Keeping the model and patient folds fixed helps us see what each field adds.

In the chart, the vertical axis lists the field combinations and the horizontal axis is held-out AP. Longer blue bars mean better ranking of malignant images. The dashed vertical line is the dummy reference; the printed values give the exact scores. Compare age alone with age plus site, then compare that with all three fields. There are no uncertainty bars here, so tiny differences should not be read as established gains.


**Historical supporting section:** its saved pre-migration diagnostic is retained below; it does not select the current final procedure.

In [ ]:
display_historical_cell(PROJECT_ROOT, 'dedup_v1', '05', 23)

Age gives the strongest individual signal, with AP 0.0412. Site alone reaches 0.0271, and adding it to age raises AP to 0.0487. Sex alone is close to the dummy, at 0.0197. Adding sex to age and site raises AP slightly to 0.0492 while lowering ROC-AUC from 0.653 to 0.643.

These are pooled AP values, matching the original chart. Primary mean-fold ROC-AUC is 0.0458 for age, 0.0636 for age plus site and 0.0648 for all three fields. Age and site account for most of the useful signal in this run. The small change from adding sex needs an uncertainty check before we treat it as a dependable improvement. These comparisons describe predictive usefulness, not the causes of melanoma.


### 6. Read the fitted age relationship

**Experiment-scope adaptation:** the original fitted a new all-development spline for this descriptive view. The clean report instead averages predictions from the five saved outer-training spline fits, so no additional model is fitted. For each age, assign that age to all records and average each saved model's scores over the clean records' sex and site, then take the equal mean across saved models. The observed-rate points use the retained development images. The performance scores above still use held-out predictions.

The horizontal axis is approximate age; the vertical axis is percentage. The gold line is this mean marginal model score; blue-outlined points are observed malignant percentages. Larger points mean more images. Compare the broad pattern: small groups can fluctuate sharply. Neither series has uncertainty intervals, and the fitted line does not establish calibrated individual risks.


**Historical supporting section:** its saved pre-migration diagnostic is retained below; it does not select the current final procedure.

In [ ]:
display_historical_cell(PROJECT_ROOT, 'dedup_v1', '05', 26)

The fitted curve generally rises at older ages. The observed rates fluctuate, particularly where few images are available: age 90 has only 16 retained images, for example. The curve smooths those jumps, but fitting a smooth line does not remove uncertainty. This is an association in the development data, not evidence that age causes an individual lesion to be malignant.


### 7. Describe the observed rates by sex and site

The next table counts malignant images within each category. Read the image count and malignant count beside the percentage. A high percentage based on a handful of cases provides much weaker evidence than the same percentage based on many cases.


In [ ]:
display_historical_cell(PROJECT_ROOT, 'dedup_v1', '05', 29)

These are unadjusted descriptive rates. Categories with few images or malignant cases, such as oral/genital and palms/soles, can show extreme percentages by chance and should not be ranked as if their estimates were equally reliable.


### 8. Choose exploratory cutoffs for confusion matrices

A cutoff turns a probability into a decision: images at or above it are flagged. At 0.50, every model in this run leaves every image unflagged. The probabilities are mostly small because malignant images are rare.

For each learned model, we try every distinct predicted probability, plus 0 and 1. At each candidate cutoff, we calculate recall and specificity, the fraction of benign images left unflagged. We choose the cutoff with the largest balanced accuracy, the average of recall and specificity. If candidates tie, we choose the one closest to 0.50. The dummy stays at 0.50 because it has no useful ranking within a fold.

The same OOF labels choose the cutoffs and assess their decisions, so this is an exploratory illustration. The training-only selection in notebook 08 provides a better check of a complete cutoff procedure.


**Historical supporting section:** its saved pre-migration diagnostic is retained below; it does not select the current final procedure.

In [ ]:
display_historical_cell(PROJECT_ROOT, 'dedup_v1', '05', 32)

### 9. Read the confusion matrices

Each panel shows the counts produced by its model's chosen cutoff. Rows are the actual class and columns are the predicted class. The upper-left cell is a benign image left unflagged (TN); the upper-right is a benign image flagged by mistake (FP). The lower-left is a missed malignant image (FN); the lower-right is a malignant image found (TP).

The percentages use the actual class as their denominator. The top row adds to 100% of benign images and the bottom row adds to 100% of malignant images. Compare the bottom-right percentage with the top-right percentage to see the detection/false-alarm trade-off. Darker blue means more images, using one shared count scale across panels. Read the numbers as well as the shading, because benign images greatly outnumber malignant ones.

These matrices make the practical cost of a cutoff visible. They do not show final-test performance or establish a clinical decision rule.


**Historical supporting section:** its saved pre-migration diagnostic is retained below; it does not select the current final procedure.

In [ ]:
display_historical_cell(PROJECT_ROOT, 'dedup_v1', '05', 34)

## Where this comparison leads

[04_metadata_baselines.ipynb](04_metadata_baselines.ipynb) gives a shorter walkthrough of the two basic references. [06_metadata_neural_network.ipynb](06_metadata_neural_network.ipynb) checks whether a small neural network helps, and [07_metadata_tuning.ipynb](07_metadata_tuning.ipynb) checks changes to the model settings.

[08_metadata_robustness.ipynb](08_metadata_robustness.ipynb) then adds patient-based uncertainty intervals, training-only cutoff selection, calibration plots and subgroup checks. The [methodology guide](../../docs/metadata_methodology.md) explains why those steps matter.


**Historical supporting section:** its saved pre-migration diagnostic is retained below; it does not select the current final procedure.

## Limitations

These folds have been used to compare several models and field combinations. Choosing the most promising result after seeing all of them can make that winner look better than it will on new patients. Patient separation prevents direct training leakage, but it cannot remove the effect of repeated development choices.

There are only 127 malignant images from 88 patients, and some ages and sites contain very few cases. This notebook does not provide confidence intervals for the model or field comparisons. Its age plot and category rates are descriptive associations; they cannot explain causation. The cutoffs are chosen and assessed on the same predictions, and the plots do not establish calibrated probabilities or clinical safety.

Metadata contains no lesion appearance. These results do not establish performance at another hospital or on the reserved test set, which has not been used to evaluate these models.


**Historical supporting section:** its saved pre-migration diagnostic is retained below; it does not select the current final procedure.

## Summary

The spline logistic model is the strongest of these five fixed methods on pooled AP (0.0492) and ROC-AUC (0.6428). Age supplies most of the individual metadata signal, and site adds information. Sex adds only a small change beyond those fields.

Its primary mean-fold ROC-AUC is 0.0648, ahead of XGBoost (0.0591), categorical logistic (0.0567) and forest (0.0548). Retain the fixed spline as the primary metadata model and the dummy as the reference. Keep the standard logistic and tree results as comparisons. At the spline's exploratory cutoff of 0.0227, it finds 65 of 127 malignant images (51.2%) while incorrectly flagging 1,484 of 6,579 benign images (22.6%). That trade-off shows why a higher ranking score alone is insufficient.

Next, check the network and tuning experiments, then use notebook 08 to assess uncertainty, probability calibration and a cutoff selected from training patients. Freeze the complete method before evaluating the reserved test set once.


**Historical supporting section:** its saved pre-migration diagnostic is retained below; it does not select the current final procedure.

## Additional deduplicated-workflow diagnostics

The complete original section sequence above is retained as the template. The following clearly labelled additions document clean artifact verification, training support and explanatory diagnostics that were not figures in the original. They remain secondary development evidence and do not select another model or cutoff.

**Historical supporting section:** its saved pre-migration diagnostic is retained below; it does not select the current final procedure.

In [ ]:
display_historical_cell(PROJECT_ROOT, 'dedup_v1', '05', 39)

In [ ]:
display_historical_cell(PROJECT_ROOT, 'dedup_v1', '05', 40)

### Added diagnostic: 7. Inspect tree importances with the proper cautions

Sum encoded-column importances back to age, sex and site. Forest values use impurity-based importance; XGBoost uses its saved normalised tree importance. Many age categories can receive more opportunities for splits. These measures are associations within fitted trees and cannot establish out-of-sample field importance or causation; compare fields within a model, not magnitudes across algorithms.

**Historical supporting section:** its saved pre-migration diagnostic is retained below; it does not select the current final procedure.

In [ ]:
display_historical_cell(PROJECT_ROOT, 'dedup_v1', '05', 42)

### Current ROC-AUC-first recap

Read the mean-fold ROC-AUC comparison first. Keep rare-positive AP, probability error, patient support and the historical diagnostics alongside it. The comparison supplies development evidence rather than a new final model or test result.

In [ ]:
render_baseline_section(report, 'summary')